# LeetCode 1682 — Longest Palindromic Subsequence II bằng Python

Link đề: https://leetcode.com/problems/longest-palindromic-subsequence-ii/ · Độ khó: **Medium** · Chủ đề: String, Dynamic Programming

> **Lưu ý (bài Premium 🔒):** đề trong notebook được viết lại theo hiểu biết về bài (nguồn tham khảo: bản mô tả của doocs/leetcode). Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def longestPalindromeSubseq(self, s: str) -> int:
        ...
```

Biến thể "khó tính" của bài 516: dãy con palindrome phải có **độ dài chẵn** và **không có hai ký tự liền nhau bằng nhau** (trừ cặp ở giữa). Notebook đi từ **liệt kê mọi dãy con** → **chọn cặp ngoài cùng rồi đệ quy vào trong** `O(n⁴)` → **đệ quy có nhớ `dfs(i, j, ký tự ngoài)`** → **quy hoạch động dưới lên `f[i][j][c]`** với mảng cuộn `O(26·n)` bộ nhớ.

## 1. Mục tiêu học

1. Hiểu điều kiện "tốt": palindrome **chẵn**, và vì đối xứng, "không có hai ký tự liền nhau bằng nhau trừ cặp giữa" ⇔ **hai cặp lồng nhau liên tiếp có ký tự khác nhau**.
2. Biết **mở rộng trạng thái** của DP palindrome kinh điển `dp[i][j]` bằng thông tin "ký tự của cặp ngoài cùng" để kiểm soát ràng buộc.
3. Cài đặt đệ quy có nhớ `dfs(i, j, x)` và DP dưới lên `f[i][j][c]`; tối ưu bộ nhớ bằng **mảng cuộn theo `i`**.
4. Hiểu cái giá của trạng thái lớn trong Python (`n² · 26` trạng thái) và cách tránh tràn bộ nhớ / quá thời gian.
5. Kiểm thử bằng liệt kê dãy con trên xâu ngắn, seed cố định.

## 2. Đề bài & ví dụ

Một dãy con của xâu `s` được gọi là **dãy con palindrome tốt** nếu:
- Nó là **dãy con** của `s` (xóa một số ký tự, giữ thứ tự).
- Nó là **palindrome** (đọc xuôi ngược như nhau).
- Nó có **độ dài chẵn**.
- **Không có hai ký tự liên tiếp bằng nhau**, **trừ** hai ký tự ở **chính giữa**.

Ví dụ với `s = "abcabcabb"`: `"abba"` là tốt; `"bcb"` không (độ dài lẻ); `"bbbb"` không (có ký tự liên tiếp bằng nhau ngoài cặp giữa).

Cho `s`, trả về **độ dài** của dãy con palindrome tốt **dài nhất**.

**Ví dụ 1:** `s = "bbabab"` → `4` (`"baab"`).

**Ví dụ 2:** `s = "dcbccacdb"` → `4` (`"dccd"`).

**Ràng buộc:**
- `1 ≤ s.length ≤ 250`
- `s` gồm chữ cái tiếng Anh in thường.

In [1]:
import random
import sys
import time
from functools import lru_cache
from itertools import combinations
from typing import List

examples = [("bbabab", 4), ("dcbccacdb", 4)]

def is_good(t):
    n = len(t)
    if n % 2 or t != t[::-1]:
        return False
    for i in range(n - 1):
        if t[i] == t[i + 1] and i != n // 2 - 1:      # chỉ cặp giữa (n/2 - 1, n/2) được bằng nhau
            return False
    return True

def is_subsequence(t, s):
    it = iter(s)
    return all(ch in it for ch in t)

for t in ["abba", "bcb", "bbbb", "", "aa", "abab", "abccba", "baab"]:
    print(f"{t!r:>9}: tốt? {is_good(t)}")
print("'baab' là dãy con của 'bbabab'?", is_subsequence("baab", "bbabab"), "| 'dccd' của 'dcbccacdb'?", is_subsequence("dccd", "dcbccacdb"))

   'abba': tốt? True
    'bcb': tốt? False
   'bbbb': tốt? False
       '': tốt? True
     'aa': tốt? True
   'abab': tốt? False
 'abccba': tốt? True
   'baab': tốt? True
'baab' là dãy con của 'bbabab'? True | 'dccd' của 'dcbccacdb'? True


Chú ý: theo định nghĩa, xâu rỗng (độ dài 0, chẵn) cũng "tốt" — nên đáp án tối thiểu là `0` (ví dụ `s = "abc"` không có cặp ký tự bằng nhau nào).

## 3. Kiến thức nền

### 3.1 Dãy con (subsequence) và liệt kê bằng tổ hợp chỉ số

Một dãy con ứng với một tập chỉ số tăng dần. `combinations(range(n), k)` liệt kê mọi tập `k` chỉ số; có tổng cộng `2ⁿ` dãy con.

In [2]:
s = "abca"
subs = [''.join(s[i] for i in idx) for k in range(len(s) + 1) for idx in combinations(range(len(s)), k)]
print(len(subs), "dãy con:", subs)

16 dãy con: ['', 'a', 'b', 'c', 'a', 'ab', 'ac', 'aa', 'bc', 'ba', 'ca', 'abc', 'aba', 'aca', 'bca', 'abca']


### 3.2 DP palindrome kinh điển (bài 516)

`dp[i][j]` = độ dài dãy con palindrome dài nhất trong `s[i..j]`:
- `s[i] == s[j]` ⇒ `dp[i][j] = dp[i+1][j−1] + 2`;
- ngược lại `dp[i][j] = max(dp[i+1][j], dp[i][j−1])`.
Duyệt `i` giảm dần, `j` tăng dần.

In [3]:
def lps(s):
    n = len(s)
    dp = [[0] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        dp[i][i] = 1
        for j in range(i + 1, n):
            dp[i][j] = dp[i + 1][j - 1] + 2 if s[i] == s[j] else max(dp[i + 1][j], dp[i][j - 1])
    return dp[0][n - 1]

print("LPS('bbbab') =", lps("bbbab"), "| LPS('bbabab') =", lps("bbabab"))

LPS('bbbab') = 4 | LPS('bbabab') = 5


### 3.3 `functools.lru_cache` / `cache` và `cache_clear`

Ghi nhớ kết quả theo tham số. Khi hàm được định nghĩa **trong** method, mỗi lần gọi method có cache riêng; nếu định nghĩa ở ngoài thì phải gọi `cache_clear()` giữa các test (hoặc đưa `s` vào tham số).

In [4]:
@lru_cache(maxsize=None)
def sq(x):
    return x * x

print(sq(12), sq.cache_info())
sq.cache_clear()
print("sau cache_clear:", sq.cache_info())

144 CacheInfo(hits=0, misses=1, maxsize=None, currsize=1)
sau cache_clear: CacheInfo(hits=0, misses=0, maxsize=None, currsize=0)


### 3.4 Đổi ký tự sang chỉ số `0..25`

Lưu `f[c]` cho 26 chữ cái bằng list, chỉ số `ord(ch) − 97`. Một "ký tự rỗng" (chưa có cặp ngoài) có thể mã hóa là `26`.

In [5]:
print([ord(ch) - 97 for ch in "azb"], "| ký tự rỗng ->", 26)

[0, 25, 1] | ký tự rỗng -> 26


## 4. Ý tưởng nền tảng / trực giác

Một palindrome chẵn được tạo bởi các **cặp lồng nhau** từ ngoài vào trong: `(c₁ … c₁)`, bên trong là `(c₂ … c₂)`, …, cặp giữa `c_m c_m`. Ví dụ `"baab"` = cặp `b` bao cặp `a`.

Viết ra: `c₁ c₂ … c_m c_m … c₂ c₁`. Hai ký tự liên tiếp bằng nhau ⇔ `cₜ = cₜ₊₁` (ở nửa trái hoặc nửa phải) — trừ `c_m c_m` ở giữa. Vậy điều kiện "tốt" ⇔ **mọi cặp lồng nhau liên tiếp có ký tự khác nhau**: `cₜ ≠ cₜ₊₁`.

Tính tay `s = "bbabab"` (chỉ số `0..5`):
- Cặp `b` ngoài cùng: chọn `s[0] = b` và `s[5] = b`. Bên trong `s[1..4] = "baba"`, cần cặp **khác `b`**: cặp `a` = `s[2], s[4]` ⇒ `"baab"`, độ dài 4.
- Có thể dài 6 không? Cần 3 cặp `c₁ ≠ c₂ ≠ c₃` lồng nhau — chỉ có 2 loại chữ `a, b` nên phải là `b a b` hoặc `a b a`: `"babbab"`/`"abaaba"`… không phải dãy con của `s` ⇒ đáp án 4.

In [6]:
def show_pairs(t):
    n = len(t)
    pairs = [(t[k], t[n - 1 - k]) for k in range(n // 2)]
    chars = [p[0] for p in pairs]
    ok = all(chars[k] != chars[k + 1] for k in range(len(chars) - 1))
    print(f"{t!r}: các cặp từ ngoài vào {chars}, liên tiếp khác nhau: {ok}, is_good: {is_good(t)}")

for t in ["baab", "dccd", "abba", "abaaba", "bbbb", "abccba"]:
    show_pairs(t)

'baab': các cặp từ ngoài vào ['b', 'a'], liên tiếp khác nhau: True, is_good: True
'dccd': các cặp từ ngoài vào ['d', 'c'], liên tiếp khác nhau: True, is_good: True
'abba': các cặp từ ngoài vào ['a', 'b'], liên tiếp khác nhau: True, is_good: True
'abaaba': các cặp từ ngoài vào ['a', 'b', 'a'], liên tiếp khác nhau: True, is_good: True
'bbbb': các cặp từ ngoài vào ['b', 'b'], liên tiếp khác nhau: False, is_good: False
'abccba': các cặp từ ngoài vào ['a', 'b', 'c'], liên tiếp khác nhau: True, is_good: True


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi dãy con

**Ý tưởng:** thử các độ dài chẵn từ lớn xuống nhỏ; với mỗi tập chỉ số, kiểm `is_good`. Dừng ở độ dài lớn nhất tìm được.

In [7]:
def lpsii_brute(s):
    n = len(s)
    for L in range(n - n % 2, 0, -2):                 # độ dài chẵn, lớn trước
        for idx in combinations(range(n), L):
            if is_good(''.join(s[i] for i in idx)):
                return L
    return 0

for s_, exp in examples:
    print(s_, "->", lpsii_brute(s_), "(kỳ vọng", exp, ")")

bbabab -> 4 (kỳ vọng 4 )
dcbccacdb -> 4 (kỳ vọng 4 )


**Độ phức tạp:** `O(2ⁿ · n)` — với `n = 250` là bất khả thi; chỉ dùng cho `n ≤ 14` để đối chiếu.

**Nút thắt:** dãy con palindrome có cấu trúc **lồng nhau** — chọn cặp ngoài cùng rồi bài toán con là phần **bên trong**, chỉ cần nhớ ký tự của cặp ngoài để không lặp lại.

## 6. Ý tưởng thuật toán / bước đột phá

**Trạng thái:** `dfs(i, j, x)` = độ dài dãy con tốt dài nhất trong `s[i..j]` mà **cặp ngoài cùng của nó khác `x`** (`x` là ký tự của cặp bao bên ngoài; `x = ''` nếu chưa có).

**Chuyển trạng thái:**
- **Nếu** `i ≥ j` **thì** `0` (không đủ hai ký tự).
- **Nếu** `s[i] == s[j]` và `s[i] ≠ x` **thì** dùng cặp `(i, j)`: `2 + dfs(i + 1, j − 1, s[i])`.
- **Ngược lại** bỏ một đầu: `max(dfs(i + 1, j, x), dfs(i, j − 1, x))`.

Vì sao được "tham lam" chọn cặp `(i, j)` ngay khi `s[i] == s[j] ≠ x` mà **không** xét nhánh bỏ một đầu? Trực giác: bất kỳ lời giải nào nằm trong `s[i..j]` đều có thể được "bọc" thêm (hoặc thay cặp ngoài cùng) bằng cặp `(i, j)` mà vẫn tốt và không ngắn đi. Chứng minh chặt ở mục 10.

**Phiên bản dưới lên:** `f[i][j][c]` = độ dài dài nhất trong `s[i..j]` với cặp ngoài cùng **đúng bằng** `c` (0 nếu không có):
- `f[i][j][c] = max(f[i+1][j][c], f[i][j−1][c])`;
- **nếu** `s[i] = s[j] = c` **thì** thêm ứng viên `2 + max_{c' ≠ c} f[i+1][j−1][c']`.
Đáp án `max_c f[0][n−1][c]`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Chọn cặp ngoài cùng `(i, j)`, đệ quy vào trong `O(n⁴)`

`g(i, j)` (với `s[i] = s[j]`, `i < j`) = độ dài dãy con tốt dài nhất **có cặp ngoài cùng đúng là `(i, j)`** = `2 + max g(k, l)` trên mọi `i < k < l < j` với `s[k] = s[l] ≠ s[i]` (hoặc `2` nếu không có). Đáp án `max g(i, j)` (hoặc 0).

In [8]:
def lpsii_outer_pairs(s):
    n = len(s)
    g = {}
    # xử lý theo độ dài khoảng tăng dần để các khoảng bên trong đã có kết quả
    for length in range(2, n + 1):
        for i in range(0, n - length + 1):
            j = i + length - 1
            if s[i] != s[j]:
                continue
            best = 2
            for (k, l), v in g.items():
                if i < k and l < j and s[k] != s[i]:
                    best = max(best, 2 + v)
            g[(i, j)] = best
    return max(g.values(), default=0)

for s_, exp in examples:
    print(s_, "->", lpsii_outer_pairs(s_), "(kỳ vọng", exp, ")")

bbabab -> 4 (kỳ vọng 4 )
dcbccacdb -> 4 (kỳ vọng 4 )


**Độ phức tạp:** tối đa `O(n²)` cặp `(i, j)`, mỗi cặp quét mọi cặp đã tính `O(n²)` ⇒ `O(n⁴)`; bộ nhớ `O(n²)`. Đúng nhưng chậm (`n = 250` ⇒ `~10⁹`). Nút thắt: ta quét lại mọi cặp bên trong; DP theo khoảng `(i, j, x)` tái sử dụng kết quả.

### 7.2 Cách 3 — Đệ quy có nhớ `dfs(i, j, x)`

In [9]:
def lpsii_memo(s):
    @lru_cache(maxsize=None)
    def dfs(i, j, x):
        if i >= j:
            return 0
        if s[i] == s[j] and s[i] != x:
            return dfs(i + 1, j - 1, s[i]) + 2
        return max(dfs(i + 1, j, x), dfs(i, j - 1, x))
    ans = dfs(0, len(s) - 1, '')
    dfs.cache_clear()
    return ans

for s_, exp in examples:
    print(s_, "->", lpsii_memo(s_), "(kỳ vọng", exp, ")")

bbabab -> 4 (kỳ vọng 4 )
dcbccacdb -> 4 (kỳ vọng 4 )


**Độ phức tạp:** số trạng thái `O(n² · 27)`, mỗi trạng thái `O(1)` ⇒ `O(27 n²)` ≈ `1.7·10⁶` với `n = 250`. Độ sâu đệ quy `≤ n` (`j − i` giảm mỗi lần) — an toàn với `n = 250`. Nhưng trong Python, `lru_cache` với hàng triệu khóa tốn **nhiều bộ nhớ** (vài trăm MB) và chậm; bản dưới lên ở mục sau tốt hơn.

### 7.3 Cách 4 — DP dưới lên `f[i][j][c]` (bảng đầy đủ)

In [10]:
def lpsii_table(s):
    n = len(s)
    if n < 2:
        return 0
    a = [ord(ch) - 97 for ch in s]
    zero = [0] * 26
    f = [[zero] * n for _ in range(n + 1)]             # f[i][j] là list 26 phần tử (chia sẻ 'zero' khi chưa tính)
    for i in range(n - 1, -1, -1):
        for j in range(i + 1, n):
            down, left = f[i + 1][j], f[i][j - 1]
            row = [x if x > y else y for x, y in zip(down, left)]
            if a[i] == a[j]:
                c = a[i]
                inner = f[i + 1][j - 1] if i + 1 <= j - 1 else zero
                best_other = max((v for k, v in enumerate(inner) if k != c), default=0)
                if best_other + 2 > row[c]:
                    row[c] = best_other + 2
            f[i][j] = row
    return max(f[0][n - 1])

for s_, exp in examples:
    print(s_, "->", lpsii_table(s_), "(kỳ vọng", exp, ")")

bbabab -> 4 (kỳ vọng 4 )
dcbccacdb -> 4 (kỳ vọng 4 )


Chú ý: `f[i][j]` với `j ≤ i` là `zero` (không có cặp nào). `zero` dùng chung một list nhưng ta **không bao giờ sửa** nó (mỗi `row` là list mới) nên an toàn. **Độ phức tạp:** `O(26 n²)` thời gian, `O(26 n²)` bộ nhớ (≈ `1.6·10⁶` số với `n = 250`).

### 7.4 Cách 5 — DP dưới lên với mảng cuộn theo `i` (lời giải chính)

`f[i][·]` chỉ cần `f[i+1][·]` (hàng dưới) và `f[i][j−1]` (cùng hàng, ô bên trái). Giữ hai hàng `prev = f[i+1]`, `cur = f[i]` ⇒ bộ nhớ `O(26 n)`.

In [11]:
def lpsii_rolling(s):
    n = len(s)
    a = [ord(ch) - 97 for ch in s]
    zero = [0] * 26
    prev = [zero] * n                       # hàng i+1
    for i in range(n - 1, -1, -1):
        cur = [zero] * n                    # hàng i; cur[j] với j <= i giữ 'zero'
        for j in range(i + 1, n):
            down, left = prev[j], cur[j - 1]
            row = [x if x > y else y for x, y in zip(down, left)]
            if a[i] == a[j]:
                c = a[i]
                inner = prev[j - 1]         # f[i+1][j-1] (là 'zero' nếu j - 1 <= i + 1 - 1 ... tức khoảng rỗng)
                best_other = 0
                for k in range(26):
                    if k != c and inner[k] > best_other:
                        best_other = inner[k]
                if best_other + 2 > row[c]:
                    row[c] = best_other + 2
            cur[j] = row
        prev = cur
    return max(prev[n - 1]) if n >= 2 else 0

for s_, exp in examples:
    print(s_, "->", lpsii_rolling(s_), "(kỳ vọng", exp, ")")

bbabab -> 4 (kỳ vọng 4 )
dcbccacdb -> 4 (kỳ vọng 4 )


`prev[j − 1]` khi `j − 1 ≤ i` (tức `j = i + 1`) là `f[i+1][i]` — khoảng rỗng, đúng là `zero`. **Độ phức tạp:** `O(26 n²)` thời gian, `O(26 n)` bộ nhớ.

## 8. TRACE từng bước

Trace cách 3 (`dfs`) trên `s = "bbabab"` — in các lời gọi **thực sự tính** (không tính cache):

In [12]:
def trace_memo(s):
    memo = {}
    def dfs(i, j, x, d):
        key = (i, j, x)
        if key in memo:
            return memo[key]
        if i >= j:
            return 0
        if s[i] == s[j] and s[i] != x:
            res = dfs(i + 1, j - 1, s[i], d + 1) + 2
            how = f"dùng cặp ({i},{j})='{s[i]}'"
        else:
            res = max(dfs(i + 1, j, x, d + 1), dfs(i, j - 1, x, d + 1))
            how = "bỏ một đầu" + (f" (s[i]=s[j]='{s[i]}' trùng x)" if s[i] == s[j] else "")
        memo[key] = res
        if d <= 3:
            print("  " * d + f"dfs({i},{j},x='{x}') s[i..j]='{s[i:j + 1]}' -> {res}  [{how}]")
        return res
    ans = dfs(0, len(s) - 1, '', 0)
    print("Số trạng thái đã tính:", len(memo), "| đáp án:", ans)
    return ans

trace_memo("bbabab")

    dfs(2,4,x='b') s[i..j]='aba' -> 2  [dùng cặp (2,4)='a']
      dfs(2,3,x='b') s[i..j]='ab' -> 0  [bỏ một đầu]
      dfs(1,2,x='b') s[i..j]='ba' -> 0  [bỏ một đầu]
    dfs(1,3,x='b') s[i..j]='bab' -> 0  [bỏ một đầu (s[i]=s[j]='b' trùng x)]
  dfs(1,4,x='b') s[i..j]='baba' -> 2  [bỏ một đầu]
dfs(0,5,x='') s[i..j]='bbabab' -> 4  [dùng cặp (0,5)='b']
Số trạng thái đã tính: 6 | đáp án: 4


4

Trace cách 4 (bảng) trên ví dụ 2 `s = "dcbccacdb"`: in giá trị `max_c f[i][j][c]` cho các khoảng, và chi tiết `f[0][8]` theo từng chữ:

In [13]:
def trace_table(s):
    n = len(s)
    a = [ord(ch) - 97 for ch in s]
    zero = [0] * 26
    f = [[zero] * n for _ in range(n + 1)]
    for i in range(n - 1, -1, -1):
        for j in range(i + 1, n):
            row = [x if x > y else y for x, y in zip(f[i + 1][j], f[i][j - 1])]
            if a[i] == a[j]:
                c = a[i]; inner = f[i + 1][j - 1] if i + 1 <= j - 1 else zero
                bo = max((v for k, v in enumerate(inner) if k != c), default=0)
                row[c] = max(row[c], bo + 2)
            f[i][j] = row
    print("bảng max_c f[i][j] (hàng i, cột j):")
    print("     " + " ".join(f"{ch:>2}" for ch in s))
    for i in range(n):
        print(f"{s[i]} {i:>2} " + " ".join(f"{max(f[i][j]) if j > i else '.':>2}" for j in range(n)))
    print("f[0][n-1] theo chữ:", {chr(97 + c): v for c, v in enumerate(f[0][n - 1]) if v})
    return max(f[0][n - 1])

trace_table("dcbccacdb")

bảng max_c f[i][j] (hàng i, cột j):
      d  c  b  c  c  a  c  d  b
d  0  .  0  0  2  2  2  2  4  4
c  1  .  .  0  2  2  2  2  2  4
b  2  .  .  .  0  2  2  2  2  4
c  3  .  .  .  .  2  2  2  2  2
c  4  .  .  .  .  .  0  2  2  2
a  5  .  .  .  .  .  .  0  0  0
c  6  .  .  .  .  .  .  .  0  0
d  7  .  .  .  .  .  .  .  .  0
b  8  .  .  .  .  .  .  .  .  .
f[0][n-1] theo chữ: {'b': 4, 'c': 2, 'd': 4}


4

Mỗi giá trị in ở dòng cuối là độ dài dãy con tốt dài nhất của cả xâu **có cặp ngoài cùng là chữ đó** (ví dụ `d → 4` ứng với `"dccd"`); đáp án là giá trị lớn nhất trong các chữ.

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def longestPalindromeSubseq(self, s: str) -> int:
        n = len(s)
        a = [ord(ch) - 97 for ch in s]
        zero = [0] * 26
        prev = [zero] * n                     # prev[j][c] = f[i+1][j][c]
        for i in range(n - 1, -1, -1):
            cur = [zero] * n                  # cur[j][c] = f[i][j][c]
            ai = a[i]
            for j in range(i + 1, n):
                row = [x if x > y else y for x, y in zip(prev[j], cur[j - 1])]
                if ai == a[j]:
                    inner = prev[j - 1]
                    best_other = 0
                    for k in range(26):
                        if k != ai and inner[k] > best_other:
                            best_other = inner[k]
                    if best_other + 2 > row[ai]:
                        row[ai] = best_other + 2
                cur[j] = row
            prev = cur
        return max(prev[n - 1]) if n >= 2 else 0

for s_, exp in examples:
    assert Solution().longestPalindromeSubseq(s_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Đặc trưng "tốt".** Dãy con palindrome chẵn `t = c₁ … c_m c_m … c₁` là tốt ⇔ `cₖ ≠ cₖ₊₁` với mọi `k < m` (mục 4). Vậy một dãy con tốt là một chuỗi cặp chỉ số lồng nhau `i₁ < i₂ < … < i_m < j_m < … < j₁` với `s[iₖ] = s[jₖ] = cₖ` và `cₖ ≠ cₖ₊₁`.

**DP `f[i][j][c]` (cách 4, 5).** Định nghĩa: độ dài lớn nhất của dãy con tốt nằm trong `s[i..j]` có cặp ngoài cùng ký tự `c`. Xét lời giải tối ưu `T`:
- Nếu cặp ngoài cùng của `T` **không** là `(i, j)` thì `T` không dùng `i` hoặc không dùng `j` ⇒ `T` nằm trong `s[i+1..j]` hoặc `s[i..j−1]` ⇒ được `max(f[i+1][j][c], f[i][j−1][c])` tính tới.
- Nếu cặp ngoài cùng **là** `(i, j)` (cần `s[i] = s[j] = c`) thì phần bên trong là một dãy con tốt trong `s[i+1..j−1]` có cặp ngoài cùng `c' ≠ c` (hoặc rỗng) ⇒ `2 + max_{c'≠c} f[i+1][j−1][c']`.
Ngược lại mọi giá trị công thức tạo ra đều ứng với một dãy con tốt có thật. ⇒ công thức đúng; đáp án là `max_c f[0][n−1][c]` (hoặc 0).

**Đệ quy `dfs(i, j, x)` (cách 3).** Ta có `dfs(i, j, x) = max_{c ≠ x} f[i][j][c]` (với `x = ''` thì không loại chữ nào). Khi `s[i] = s[j] = c ≠ x`, ta chứng minh nhánh "dùng cặp `(i, j)`" có giá trị `V = 2 + max_{c' ≠ c} f[i+1][j−1][c']` **không nhỏ hơn** mọi `f[i][j][c'']` với `c'' ≠ x`:
- `c'' = c`: lời giải tốt nhất có cặp ngoài cùng ký tự `c` là `2 + (lời giải bên trong tốt nhất có ngoài cùng ≠ c)`; đẩy cặp ngoài cùng ra tận `(i, j)` chỉ làm phần bên trong rộng hơn ⇒ đúng bằng `V`.
- `c'' ≠ c`: gọi `T` là lời giải tối ưu có ngoài cùng `c''`. `T` không thể dùng chỉ số `i` (nếu dùng thì `s[i] = c` sẽ là ký tự của cặp ngoài cùng — mâu thuẫn), tương tự không dùng `j` ⇒ `T` nằm trong `s[i+1..j−1]` với ngoài cùng `c'' ≠ c` ⇒ bọc `T` bằng cặp `(i, j)` được lời giải tốt dài `|T| + 2 ≤ V`.

Vậy `dfs` bỏ qua nhánh "bỏ một đầu" trong trường hợp này mà không mất nghiệm.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`, `Σ = 26`.

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê dãy con | `O(2ⁿ · n)` | `O(n)` |
| 2. Cặp ngoài cùng | `O(n⁴)` | `O(n²)` |
| 3. Đệ quy có nhớ | `O(Σ n²)` | `O(Σ n²)` (cache) + `O(n)` stack |
| 4. Bảng đầy đủ | `O(Σ n²)` | `O(Σ n²)` |
| 5. Mảng cuộn | `O(Σ n²)` | `O(Σ n)` |

Đo thời gian (bản ghi nhớ và `O(n⁴)` chỉ chạy với `n` nhỏ hơn):

In [15]:
rnd = random.Random(1682)
s250 = ''.join(rnd.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(250))
s250_small_alpha = ''.join(rnd.choice("abc") for _ in range(250))
for name, fn in [("table", lpsii_table), ("rolling", lpsii_rolling), ("Solution", Solution().longestPalindromeSubseq)]:
    t0 = time.perf_counter(); r1 = fn(s250); r2 = fn(s250_small_alpha); dt = time.perf_counter() - t0
    print(f"{name:<9} n=250 (26 chữ & 3 chữ): {r1:>3}, {r2:>3}  {dt * 1000:7.0f} ms")
s80 = s250_small_alpha[:80]
t0 = time.perf_counter(); r = lpsii_memo(s80); dt = time.perf_counter() - t0
print(f"memo      n=80: {r} ({dt * 1000:.0f} ms) | rolling: {lpsii_rolling(s80)}")
s40 = s250_small_alpha[:40]
t0 = time.perf_counter(); r = lpsii_outer_pairs(s40); dt = time.perf_counter() - t0
print(f"outer_pairs n=40: {r} ({dt * 1000:.0f} ms) | rolling: {lpsii_rolling(s40)}")

table     n=250 (26 chữ & 3 chữ):  72, 132       89 ms


rolling   n=250 (26 chữ & 3 chữ):  72, 132       66 ms


Solution  n=250 (26 chữ & 3 chữ):  72, 132       65 ms
memo      n=80: 38 (2 ms) | rolling: 38
outer_pairs n=40: 18 (1 ms) | rolling: 18


## 12. Bộ test

In [16]:
ALL = {
    "outer_pairs": lpsii_outer_pairs,
    "memo": lpsii_memo,
    "table": lpsii_table,
    "rolling": lpsii_rolling,
    "Solution": lambda s: Solution().longestPalindromeSubseq(s),
}

def check(s, exp=None, brute=True):
    ref = lpsii_brute(s) if brute else lpsii_rolling(s)
    for name, fn in ALL.items():
        if not brute and name in ("outer_pairs",) and len(s) > 60:
            continue
        got = fn(s)
        assert got == ref, (name, s, got, ref)
    if exp is not None:
        assert ref == exp, (s, ref, exp)
    return ref

for s_, exp in examples:
    check(s_, exp)
print("Ví dụ đề: OK")

for s_, exp in [("a", 0), ("aa", 2), ("ab", 0), ("aaaa", 2), ("abba", 4), ("abab", 2), ("abcabcabb", 4),
                ("abcba", 4), ("aabbaa", 4), ("abacaba", 6)]:
    check(s_, exp)
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


Các edge case trên (trừ ví dụ đề) là **tự dựng** — kỳ vọng được xác nhận bằng liệt kê dãy con (`lpsii_brute`) ngay trong `check`; nên đối chiếu lại đề gốc nếu muốn. Ví dụ `"abcba"` → `4` (`"abba"`, bỏ chữ `c` ở giữa vì độ dài phải chẵn), `"abacaba"` → `6` (`"abaaba"`: các cặp từ ngoài vào `a, b, a` — liên tiếp khác nhau).

In [17]:
rnd = random.Random(2012)
for _ in range(1200):
    n = rnd.randint(1, 12)
    alpha = "abcd"[:rnd.randint(1, 4)]
    check(''.join(rnd.choice(alpha) for _ in range(n)))
print("1200 xâu ngẫu nhiên ngắn so với liệt kê dãy con: OK")

for _ in range(20):
    n = rnd.randint(20, 60)
    s_ = ''.join(rnd.choice("abc") for _ in range(n))
    assert lpsii_memo(s_) == lpsii_table(s_) == lpsii_rolling(s_) == Solution().longestPalindromeSubseq(s_)
print("20 xâu cỡ vừa: các cách DP khớp nhau: OK")

1200 xâu ngẫu nhiên ngắn so với liệt kê dãy con: OK
20 xâu cỡ vừa: các cách DP khớp nhau: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng nguyên DP palindrome thường (bài 516)

In [18]:
def lpsii_bug_plain(s):
    return lps(s) if s else 0                     # SAI: không bắt độ dài chẵn, không cấm cặp liền nhau trùng

for s_ in ["bbbb", "aba", "bbabab"]:
    print(f"{s_}: sai = {lpsii_bug_plain(s_)}, đúng = {Solution().longestPalindromeSubseq(s_)}")

bbbb: sai = 4, đúng = 2
aba: sai = 3, đúng = 2
bbabab: sai = 5, đúng = 4


**Vì sao sai:** bài 516 cho phép độ dài lẻ (`"aba"`) và cặp lồng liền nhau trùng ký tự (`"bbbb"`). **Sửa:** thêm ký tự cặp ngoài vào trạng thái và chỉ ghép cặp khi `s[i] == s[j]` **và** khác ký tự cặp bao ngoài; không có ký tự đơn ở giữa (`i ≥ j` trả 0).

### Lỗi 2 — Chỉ thêm điều kiện "khác ký tự ngoài" nhưng vẫn cho độ dài lẻ

In [19]:
def lpsii_bug_odd(s):
    @lru_cache(maxsize=None)
    def dfs(i, j, x):
        if i > j:
            return 0
        if i == j:
            return 1 if s[i] != x else 0           # SAI: cho phép ký tự đơn ở giữa
        if s[i] == s[j] and s[i] != x:
            return dfs(i + 1, j - 1, s[i]) + 2
        return max(dfs(i + 1, j, x), dfs(i, j - 1, x))
    return dfs(0, len(s) - 1, '')

for s_ in ["aba", "abcba"]:
    print(f"{s_}: sai = {lpsii_bug_odd(s_)}, đúng = {Solution().longestPalindromeSubseq(s_)}")

aba: sai = 3, đúng = 2
abcba: sai = 5, đúng = 4


**Vì sao sai:** dãy con tốt phải có **độ dài chẵn**. **Sửa:** `if i >= j: return 0`.

### Lỗi 3 — Ghi nhớ thiếu tham số `x` (chỉ theo `(i, j)`)

In [20]:
def lpsii_bug_memo_ij(s):
    memo = {}
    def dfs(i, j, x):
        if i >= j:
            return 0
        if (i, j) in memo:                          # SAI: kết quả phụ thuộc cả x
            return memo[(i, j)]
        if s[i] == s[j] and s[i] != x:
            res = dfs(i + 1, j - 1, s[i]) + 2
        else:
            res = max(dfs(i + 1, j, x), dfs(i, j - 1, x))
        memo[(i, j)] = res
        return res
    return dfs(0, len(s) - 1, '')

rnd = random.Random(3)
found = None
while found is None:
    t = ''.join(rnd.choice("ab") for _ in range(rnd.randint(2, 9)))
    if lpsii_bug_memo_ij(t) != Solution().longestPalindromeSubseq(t):
        found = t
print(f"phản ví dụ {found!r}: nhớ theo (i,j) = {lpsii_bug_memo_ij(found)}, đúng = {Solution().longestPalindromeSubseq(found)}")

phản ví dụ 'aabaaabaa': nhớ theo (i,j) = 4, đúng = 6


**Vì sao sai:** cùng khoảng `(i, j)` nhưng với ký tự cấm `x` khác nhau thì đáp án khác nhau. Lưu theo `(i, j)` làm lẫn kết quả. **Sửa:** khóa ghi nhớ là `(i, j, x)`.

### Lỗi 4 — `@cache` đặt ở ngoài, dùng chung giữa các xâu

In [21]:
CURRENT = {"s": ""}

@lru_cache(maxsize=None)
def dfs_global(i, j, x):
    s = CURRENT["s"]
    if i >= j:
        return 0
    if s[i] == s[j] and s[i] != x:
        return dfs_global(i + 1, j - 1, s[i]) + 2
    return max(dfs_global(i + 1, j, x), dfs_global(i, j - 1, x))

def lpsii_bug_global(s):
    CURRENT["s"] = s                                  # SAI: cache vẫn giữ kết quả của xâu trước
    return dfs_global(0, len(s) - 1, '')

print("lần 1 'abba':", lpsii_bug_global("abba"))
print("lần 2 'abcd':", lpsii_bug_global("abcd"), "| đúng:", Solution().longestPalindromeSubseq("abcd"))

lần 1 'abba': 4
lần 2 'abcd': 4 | đúng: 0


**Vì sao sai:** khóa `(0, 3, '')` đã được cache với xâu `"abba"`. Trên LeetCode, nhiều test chạy trong cùng tiến trình nên lỗi này có thật. **Sửa:** định nghĩa hàm cache **bên trong** method, hoặc gọi `cache_clear()` (như `lpsii_memo`).

### Lỗi 5 — Lấy `max` gồm cả chính ký tự `c` cho phần bên trong

In [22]:
def lpsii_bug_inner_same(s):
    n = len(s)
    a = [ord(ch) - 97 for ch in s]
    zero = [0] * 26
    prev = [zero] * n
    for i in range(n - 1, -1, -1):
        cur = [zero] * n
        for j in range(i + 1, n):
            row = [x if x > y else y for x, y in zip(prev[j], cur[j - 1])]
            if a[i] == a[j]:
                row[a[i]] = max(row[a[i]], max(prev[j - 1]) + 2)    # SAI: không loại c' == c
            cur[j] = row
        prev = cur
    return max(prev[n - 1]) if n >= 2 else 0

print("'bbbb': sai =", lpsii_bug_inner_same("bbbb"), "| đúng =", Solution().longestPalindromeSubseq("bbbb"))

'bbbb': sai = 4 | đúng = 2


**Sửa:** `max` trên `c' ≠ c`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Dãy con palindrome dài nhất" ⇒ DP khoảng `dp[i][j]`, duyệt `i` giảm, `j` tăng.
- Có **ràng buộc giữa hai cặp lồng nhau liên tiếp** ⇒ thêm "ký tự của cặp ngoài cùng" vào trạng thái (`× 26`).
- Biến các điều kiện trên **chuỗi kết quả** thành điều kiện trên **các cặp** (nhờ tính đối xứng) — ở đây "không có hai ký tự liền nhau bằng nhau" ⇔ "cặp liên tiếp khác ký tự".
- Python + trạng thái `n² · 26` ⇒ ưu tiên DP dưới lên với list, cuộn mảng để giảm bộ nhớ.
- Luôn có brute force liệt kê dãy con cho `n ≤ 12` để kiểm.

## 15. Biến thể của bài

1. **LeetCode 516 — Longest Palindromic Subsequence:** không ràng buộc chẵn/liền kề.
2. **Cho phép độ dài lẻ** (thêm ký tự giữa khác ký tự cặp trong cùng): cho `i == j` trả `1` nếu `s[i] ≠ x`.
3. **Đếm số dãy con palindrome tốt** (phân biệt theo chỉ số): DP đếm, cẩn thận trùng lặp.
4. **LeetCode 1771 — Maximize Palindrome Length From Subsequences:** palindrome từ hai xâu, phải dùng ít nhất một ký tự mỗi xâu.
5. **Yêu cầu các cặp lồng liên tiếp khác nhau ít nhất `d` (theo bảng chữ)**: cùng trạng thái, đổi điều kiện `c' ≠ c` thành `|c' − c| ≥ d`.
6. **Truy vết dãy con tốt dài nhất:** lưu lựa chọn trong DP.

In [23]:
def lpsii_allow_odd(s):
    """Biến thể 2: cho phép một ký tự giữa (khác ký tự cặp trong cùng)."""
    @lru_cache(maxsize=None)
    def dfs(i, j, x):
        if i > j:
            return 0
        if i == j:
            return 1 if s[i] != x else 0
        if s[i] == s[j] and s[i] != x:
            return max(dfs(i + 1, j - 1, s[i]) + 2, dfs(i + 1, j, x), dfs(i, j - 1, x))
        return max(dfs(i + 1, j, x), dfs(i, j - 1, x))
    return dfs(0, len(s) - 1, '')

def brute_allow_odd(s):
    def ok(t):
        if t != t[::-1]: return False
        return all(t[k] != t[k + 1] or (len(t) % 2 == 0 and k == len(t) // 2 - 1) for k in range(len(t) - 1))
    n = len(s)
    return max((L for L in range(n, -1, -1) for idx in combinations(range(n), L) if ok(''.join(s[i] for i in idx))), default=0)

rnd = random.Random(15)
for _ in range(400):
    t = ''.join(rnd.choice("abc") for _ in range(rnd.randint(1, 10)))
    assert lpsii_allow_odd(t) == brute_allow_odd(t), t
print("Biến thể cho phép độ dài lẻ: OK | 'aba' ->", lpsii_allow_odd("aba"), "| 'bbbb' ->", lpsii_allow_odd("bbbb"))

Biến thể cho phép độ dài lẻ: OK | 'aba' -> 3 | 'bbbb' -> 2


Lưu ý ở biến thể này phải xét **cả** nhánh bỏ một đầu kể cả khi `s[i] = s[j] ≠ x` (vì phần lẻ ở giữa làm lập luận "luôn dùng cặp ngoài" không còn hiển nhiên) — an toàn hơn khi không chắc.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê | mọi dãy con, kiểm `is_good` | `O(2ⁿ n)` | `O(n)` | đối chiếu, `n ≤ 14` |
| 2. Cặp ngoài cùng | `g(i,j) = 2 + max g(k,l)` | `O(n⁴)` | `O(n²)` | hiểu cấu trúc lồng nhau |
| 3. `dfs(i, j, x)` | ghi nhớ theo ký tự cấm | `O(Σn²)` | `O(Σn²)` | code ngắn, `n` vừa |
| 4. Bảng `f[i][j][c]` | DP dưới lên | `O(Σn²)` | `O(Σn²)` | dễ in/truy vết |
| 5. Mảng cuộn | chỉ giữ hàng `i+1` | `O(Σn²)` | `O(Σn)` | **nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().longestPalindromeSubseq("aabb")`, `("abccba")`, `("abcdcba")`?

**Bài 2 (tính tay):** Với `s = "abab"`, liệt kê mọi trạng thái `dfs(i, j, x)` được tính và giá trị của chúng.

**Bài 3 (điền TODO):** Hoàn thiện `reconstruct(s)` trả về **một** dãy con tốt dài nhất (chuỗi), dùng `dfs` có nhớ rồi truy vết.

**Bài 4 (tự viết lại):** Viết lại cách 5 nhưng lưu, cho mỗi `(i, j)`, chỉ **hai giá trị lớn nhất** của `f[i][j][·]` cùng ký tự tương ứng (thay vì đủ 26) — để lấy `max_{c' ≠ c}` trong `O(1)`. Kiểm với `Solution`.

**Bài 5 (mini challenge):** Biến thể khoảng cách: cặp lồng liên tiếp phải có ký tự khác nhau **ít nhất `d`** theo bảng chữ cái (`|c − c'| ≥ d`). Viết `lpsii_gap(s, d)` (với `d = 1` trùng bài gốc), kiểm bằng liệt kê.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def reconstruct(s):
    # TODO: dfs có nhớ như lpsii_memo; sau đó đi lại từ (0, n-1, '') chọn nhánh cho đúng giá trị
    pass

# Bài 4 — TODO
def lpsii_top2(s):
    # TODO: thay list 26 bằng (best1, c1, best2) với c1 là chữ của best1; f[c] cụ thể vẫn cần cho row[c]...
    pass

# Bài 5 — TODO
def lpsii_gap(s, d):
    # TODO: như dfs(i, j, x) nhưng điều kiện ghép là x == '' hoặc abs(ord(s[i]) - ord(x)) >= d
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `"aabb"` không có palindrome dài hơn 2 (`"aa"` hoặc `"bb"`).
- Mức 2: `"abccba"`: cặp `a`, `b`, `c` lồng nhau, liên tiếp khác nhau.
- Mức 3: `"abcdcba"`: độ dài lẻ không được; bỏ `d` được `"abccba"`.

**Bài 2**
- Mức 1: Bắt đầu `dfs(0, 3, '')`: `s[0] = a ≠ s[3] = b`.
- Mức 2: Bỏ một đầu: `dfs(1, 3, '')` và `dfs(0, 2, '')`.
- Mức 3: `dfs(1, 3, '')` dùng cặp `b`: `2 + dfs(2, 2, 'b') = 2`.

**Bài 3**
- Mức 1: Ở trạng thái dùng cặp: thêm `s[i]` vào hai đầu.
- Mức 2: Ở trạng thái bỏ đầu: đi theo nhánh có giá trị bằng `dfs(i, j, x)`.
- Mức 3: Kiểm `is_good` và `is_subsequence`, độ dài bằng đáp án.

**Bài 4**
- Mức 1: `max_{c' ≠ c}` = `best1` nếu `c1 ≠ c`, ngược lại `best2`.
- Mức 2: Vẫn cần `f[i][j][c]` cho từng `c`… nên lưu thêm list nhưng tính top-2 kèm theo.
- Mức 3: Cập nhật top-2 sau khi tạo `row`.

**Bài 5**
- Mức 1: Chỉ đổi điều kiện ghép cặp.
- Mức 2: `d = 0` ⇒ thành bài palindrome chẵn thường (cho phép trùng).
- Mức 3: Brute force: kiểm điều kiện trên dãy các cặp.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
r1 = [Solution().longestPalindromeSubseq(t) for t in ("aabb", "abccba", "abcdcba")]
print("Bài 1:", r1)
assert r1 == [2, 6, 6] == [lpsii_brute(t) for t in ("aabb", "abccba", "abcdcba")]

# Bài 2
trace_memo("abab")

# Bài 3
def reconstruct_sol(s):
    @lru_cache(maxsize=None)
    def dfs(i, j, x):
        if i >= j:
            return 0
        if s[i] == s[j] and s[i] != x:
            return dfs(i + 1, j - 1, s[i]) + 2
        return max(dfs(i + 1, j, x), dfs(i, j - 1, x))
    left = []
    i, j, x = 0, len(s) - 1, ''
    while i < j:
        if s[i] == s[j] and s[i] != x:
            left.append(s[i]); x = s[i]; i += 1; j -= 1
        elif dfs(i + 1, j, x) == dfs(i, j, x):
            i += 1
        else:
            j -= 1
    return ''.join(left) + ''.join(reversed(left))

rnd = random.Random(3)
for _ in range(800):
    t = ''.join(rnd.choice("abc") for _ in range(rnd.randint(1, 12)))
    r = reconstruct_sol(t)
    assert is_good(r) and is_subsequence(r, t) and len(r) == Solution().longestPalindromeSubseq(t)
print("Bài 3 OK | ví dụ:", reconstruct_sol("bbabab"), reconstruct_sol("dcbccacdb"))

Bài 1: [2, 6, 6]
  dfs(1,3,x='') s[i..j]='bab' -> 2  [dùng cặp (1,3)='b']
  dfs(0,2,x='') s[i..j]='aba' -> 2  [dùng cặp (0,2)='a']
dfs(0,3,x='') s[i..j]='abab' -> 2  [bỏ một đầu]
Số trạng thái đã tính: 3 | đáp án: 2
Bài 3 OK | ví dụ: baab bccb


In [26]:
# Bài 4
def lpsii_top2_sol(s):
    n = len(s)
    a = [ord(ch) - 97 for ch in s]
    zero = ([0] * 26, (0, -1, 0))                        # (list f[c], (best1, c1, best2))
    prev = [zero] * n
    for i in range(n - 1, -1, -1):
        cur = [zero] * n
        for j in range(i + 1, n):
            row = [x if x > y else y for x, y in zip(prev[j][0], cur[j - 1][0])]
            if a[i] == a[j]:
                b1, c1, b2 = prev[j - 1][1]
                other = b1 if c1 != a[i] else b2        # max_{c' != c} trong O(1)
                if other + 2 > row[a[i]]:
                    row[a[i]] = other + 2
            # tính top-2 của row
            b1 = b2 = 0; c1 = -1
            for c, v in enumerate(row):
                if v > b1:
                    b2, b1, c1 = b1, v, c
                elif v > b2:
                    b2 = v
            cur[j] = (row, (b1, c1, b2))
        prev = cur
    return max(prev[n - 1][0]) if n >= 2 else 0

rnd = random.Random(4)
for _ in range(600):
    t = ''.join(rnd.choice("abcd") for _ in range(rnd.randint(1, 30)))
    assert lpsii_top2_sol(t) == Solution().longestPalindromeSubseq(t)
print("Bài 4 OK")

# Bài 5
def lpsii_gap_sol(s, d):
    @lru_cache(maxsize=None)
    def dfs(i, j, x):
        if i >= j:
            return 0
        best = max(dfs(i + 1, j, x), dfs(i, j - 1, x))
        if s[i] == s[j] and (x == '' or abs(ord(s[i]) - ord(x)) >= d):
            best = max(best, dfs(i + 1, j - 1, s[i]) + 2)
        return best
    return dfs(0, len(s) - 1, '')

def lpsii_gap_brute(s, d):
    n = len(s)
    for L in range(n - n % 2, 0, -2):
        for idx in combinations(range(n), L):
            t = ''.join(s[i] for i in idx)
            if t == t[::-1]:
                chars = t[:L // 2]
                if all(abs(ord(chars[k]) - ord(chars[k + 1])) >= d for k in range(len(chars) - 1)):
                    return L
    return 0

rnd = random.Random(5)
for _ in range(500):
    t = ''.join(rnd.choice("abcde") for _ in range(rnd.randint(1, 10)))
    d = rnd.randint(0, 3)
    assert lpsii_gap_sol(t, d) == lpsii_gap_brute(t, d), (t, d)
assert all(lpsii_gap_sol(t, 1) == Solution().longestPalindromeSubseq(t) for t in ["bbabab", "dcbccacdb", "abcabcabb"])
print("Bài 5 OK | 'acbbca' với d=2:", lpsii_gap_sol("acbbca", 2), "| d=1:", lpsii_gap_sol("acbbca", 1))

Bài 4 OK
Bài 5 OK | 'acbbca' với d=2: 4 | d=1: 6


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template DP khoảng có "ký tự cặp ngoài cùng" (dưới lên, mảng cuộn):**

```python
zero = [0] * 26
prev = [zero] * n
for i in range(n - 1, -1, -1):
    cur = [zero] * n
    for j in range(i + 1, n):
        row = [max(x, y) for x, y in zip(prev[j], cur[j - 1])]      # bỏ một đầu
        if s[i] == s[j]:
            c = code(s[i])
            other = max(v for k, v in enumerate(prev[j - 1]) if k != c)
            row[c] = max(row[c], other + 2)                          # dùng cặp (i, j)
        cur[j] = row
    prev = cur
return max(prev[n - 1]) if n >= 2 else 0
```

**Template đệ quy:** `dfs(i, j, x)`: `i ≥ j → 0`; `s[i] == s[j] != x → 2 + dfs(i+1, j−1, s[i])`; ngược lại `max(dfs(i+1, j, x), dfs(i, j−1, x))`.

**Checklist:**
- [ ] Độ dài **chẵn** (trường hợp `i == j` trả 0)?
- [ ] Cặp lồng liên tiếp phải **khác** ký tự (loại `c' == c`)?
- [ ] Khóa ghi nhớ gồm cả `x`?
- [ ] Cache không dùng chung giữa các test?
- [ ] Bộ nhớ: DP dưới lên với mảng cuộn cho `n = 250`?
- [ ] Đã so với liệt kê dãy con trên xâu ngắn?
- [ ] Chữ ký đúng: `def longestPalindromeSubseq(self, s: str) -> int`?